# Bomberland PPO Training

Checkpoints are saved to `/kaggle/working/ckpts/` — download from the Output tab when done.

In [ ]:
%%bash
pip install stable-baselines3>=2.2.1 gymnasium>=0.29.1 --quiet

In [ ]:
%%bash
set -e

# Clone our project (ppo_agent/ lives here)
if [ ! -d /kaggle/working/project ]; then
    git clone https://github.com/PhuDoan23/GDGoC-AI-Challenge-2026 /kaggle/working/project --quiet
    echo "Project cloned"
else
    cd /kaggle/working/project && git pull --quiet
    echo "Project updated"
fi

# Clone participant kit INSIDE project/ so gym_wrapper.py finds it via relative path
if [ ! -d /kaggle/working/project/kit ]; then
    git clone https://github.com/VLTisME/Bomberland-GDGoC-AI-Challenge /kaggle/working/project/kit --quiet
    echo "Kit cloned into project/kit/"
else
    echo "Kit already present"
fi

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}  (sm_{cap[0]}{cap[1]})")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    if cap[0] < 7:
        print("WARNING: This GPU is not supported by PyTorch 2.1+ — will train on CPU")

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────
PHASE        = "warmup"      # "warmup" or "league"
TOTAL_STEPS  = 5_000_000
N_ENVS       = 8
N_EPOCHS     = 4
N_STEPS      = 2048
BATCH_SIZE   = 64
LOAD_CHECKPOINT = None       # e.g. "/kaggle/working/ckpts/warmup_final"
CKPT_DIR = "/kaggle/working/ckpts"
LOG_DIR  = "/kaggle/working/logs"
# ───────────────────────────────────────────────────────────────────────────────

import os
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
print(f"Phase: {PHASE}  |  Steps: {TOTAL_STEPS:,}")

In [ ]:
import sys

PROJECT_DIR = "/kaggle/working/project"
PPO_DIR     = f"{PROJECT_DIR}/ppo_agent"
KIT_DIR     = f"{PROJECT_DIR}/kit"   # matches gym_wrapper.py's relative path

for d in (KIT_DIR, PPO_DIR, PROJECT_DIR):
    if d not in sys.path:
        sys.path.insert(0, d)

from engine.game import BomberEnv
from encode_obs import encode_obs
from gym_wrapper import BomberGymEnv, LeaguePool
from model import make_ppo, _best_device

device = _best_device()
print(f"Training device: {device}")

env = BomberEnv(seed=0)
obs = env.reset(seed=0)
s, a = encode_obs(obs, 0)
print(f"obs: map={s.shape}  aux={a.shape}  — engine OK")

In [ ]:
import time
from stable_baselines3.common.vec_env import DummyVecEnv

pool = LeaguePool(["random", "simple"])
vec_env = DummyVecEnv([lambda: BomberGymEnv(pool) for _ in range(N_ENVS)])
test_model = make_ppo(vec_env, device=device, verbose=0,
                      n_epochs=N_EPOCHS, n_steps=N_STEPS, batch_size=BATCH_SIZE)

N_BENCH = N_STEPS * N_ENVS * 2
t0 = time.time()
test_model.learn(total_timesteps=N_BENCH)
fps = N_BENCH / (time.time() - t0)

print(f"FPS: {fps:.0f} steps/sec")
print(f"ETA this run ({TOTAL_STEPS/1e6:.0f}M steps): {TOTAL_STEPS/fps/3600:.1f} hrs")
vec_env.close()

In [ ]:
cmd_parts = [
    f"cd {PROJECT_DIR} &&",
    "python3 ppo_agent/train.py",
    f"--phase {PHASE}",
    f"--total_steps {TOTAL_STEPS}",
    f"--n_envs {N_ENVS}",
    f"--n_epochs {N_EPOCHS}",
    f"--n_steps {N_STEPS}",
    f"--batch_size {BATCH_SIZE}",
    f"--device {device}",
    f"--ckpt_dir {CKPT_DIR}",
    f"--log_dir {LOG_DIR}",
]
if LOAD_CHECKPOINT:
    cmd_parts.append(f"--load_checkpoint {LOAD_CHECKPOINT}")

cmd = " ".join(cmd_parts)
print("Running:\n", cmd)
os.system(cmd)

In [ ]:
print("\n=== Saved checkpoints ===")
for f in sorted(os.listdir(CKPT_DIR)):
    size_mb = os.path.getsize(os.path.join(CKPT_DIR, f)) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")

In [ ]:
import glob
from train import eval_vs_baseline
from stable_baselines3 import PPO

actor_ckpts = glob.glob(f"{CKPT_DIR}/*actor*.pth")
if not actor_ckpts:
    print("No actor checkpoint found — skipping eval")
else:
    pool2 = LeaguePool(["random", "simple"])
    vec2  = DummyVecEnv([lambda: BomberGymEnv(pool2)])
    final_model = PPO.load(f"{CKPT_DIR}/{PHASE}_final", env=vec2, device=device)

    print(f"=== Final eval ({PHASE}) ===")
    for baseline in ["random", "simple", "smarter", "genius", "tactical"]:
        stats = eval_vs_baseline(final_model, baseline, n_games=30)
        print(f"  vs {baseline:12s}: {stats['win_rate']:5.1%}  "
              f"(W={stats['wins']} D={stats['draws']} L={stats['losses']})")
    vec2.close()

## Next steps
1. **Download**: Output tab → `ckpts/warmup_actor_final.pth`
2. **Pack locally**: `python3 scripts/pack_submission.py --checkpoint ckpts/warmup_actor_final.pth`
3. **League run**: Set `PHASE="league"`, `TOTAL_STEPS=20_000_000`, `LOAD_CHECKPOINT="/kaggle/working/ckpts/warmup_final"`, re-run